In [3]:
# import the best parameters from xgb_best_params.json

import json
with open("xgb_best_params.json", "r") as f:
    best_params = json.load(f)  
print("Best parameters:", best_params)

best_params_price = best_params["price"]
best_params_news = best_params["news"]
best_params_emb = best_params["embeddings"]


Best parameters: {'price': {'max_depth': 6, 'min_child_weight': 9.414942824935029, 'eta': 0.010103505214852922, 'subsample': 0.8443464008318593, 'colsample_bytree': 0.6405480132883279, 'lambda': 0.00025079252162247986, 'alpha': 0.4157661596047824}, 'news': {'max_depth': 7, 'min_child_weight': 8.763348041314794, 'eta': 0.00863982402503789, 'subsample': 0.9316621146144122, 'colsample_bytree': 0.7836354867728363, 'lambda': 4.179887618934322, 'alpha': 2.8674000588765827}, 'embeddings': {'max_depth': 5, 'min_child_weight': 13.526369920891472, 'eta': 0.0060256031864824615, 'lambda': 0.7986934964547976, 'alpha': 1.9278630482302592, 'subsample': 0.9104958966643895, 'colsample_bytree': 0.644726102648078}}


In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

# ----------------------------------------
# 0. Reload clean data
# ----------------------------------------
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"]).reset_index(drop=True)
val   = val.sort_values(["tic", "Date"]).reset_index(drop=True)
test  = test.sort_values(["tic", "Date"]).reset_index(drop=True)

news_features = ["mean_sentiment", "max_sentiment", "min_sentiment",
                 "sum_sentiment", "news_count"]

emb_cols = [c for c in train.columns if c.startswith("pca_emb_")]
print("Embedding columns:", len(emb_cols))

def add_target(df):
    df = df.sort_values(["tic", "Date"]).copy()
    grp_close = df.groupby("tic")["Close"]
    df["ret_1d_raw"] = grp_close.pct_change(1)
    df["target_1d"] = df.groupby("tic")["ret_1d_raw"].shift(-1)
    return df

train = add_target(train)
val   = add_target(val)
test  = add_target(test)

def add_price_features(df):
    df = df.sort_values(["tic","Date"]).copy()

    grp_close = df.groupby("tic")["Close"]
    grp_ret   = df.groupby("tic")["ret_1d_raw"]
    grp_vol   = df.groupby("tic")["Volume"]

    # Price momentum (best-performing set)
    df["ret_1d_lag"] = grp_close.pct_change(1)
    df["ret_2d"]     = grp_close.pct_change(2)
    df["ret_3d"]     = grp_close.pct_change(3)
    df["ret_5d"]     = grp_close.pct_change(5)
    df["ret_10d"]    = grp_close.pct_change(10)

    # Intraday pieces
    prev_close = grp_close.shift(1)
    df["overnight_ret"] = df["Open"] / prev_close - 1
    df["intraday_ret"]  = df["Close"] / df["Open"] - 1

    # RSI
    delta = grp_close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    roll_up = gain.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    roll_down = loss.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    df["RSI"] = 100 - (100 / (1 + roll_up / roll_down))

    # Volatility (best set)
    df["vol_3d"]  = grp_ret.rolling(3).std().reset_index(0, drop=True)
    df["vol_5d"]  = grp_ret.rolling(5).std().reset_index(0, drop=True)
    df["vol_10d"] = grp_ret.rolling(10).std().reset_index(0, drop=True)

    # High-Low range volatility
    df["range_vol"] = np.log(df["High"] / df["Low"]) ** 2

    # Liquidity
    mean_vol = grp_vol.transform("mean")
    std_vol  = grp_vol.transform("std")
    df["turnover"] = df["Volume"] / mean_vol
    df["vol_z"]    = (df["Volume"] - mean_vol) / std_vol
    df["amihud"]   = (np.abs(df["ret_1d_raw"]) / df["Volume"]).replace(np.inf, np.nan)

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df

train = add_price_features(train)
val   = add_price_features(val)
test  = add_price_features(test)

def add_news_features(df):
    df = df.sort_values(["tic","Date"]).copy()
    grp = df.groupby("tic")

    for f in news_features:
        # Same-day sentiment already available: df[f]

        # lagged sentiment
        df[f + "_lag1"] = grp[f].shift(1)

        # rolling windows ending at t (leak-free)
        df[f + "_roll3"] = grp[f].rolling(3).mean().reset_index(0, drop=True)
        df[f + "_roll5"] = grp[f].rolling(5).mean().reset_index(0, drop=True)

        # sentiment volatility
        df[f + "_vol5"] = grp[f].rolling(5).std().reset_index(0, drop=True)

        # sentiment shocks
        df[f + "_shock"] = df[f] - df[f + "_lag1"]

        # sentiment surprise vs short-term mean
        df[f + "_surprise3"] = df[f] - df[f + "_roll3"]

    return df.fillna(0.0)

train = add_news_features(train)
val   = add_news_features(val)
test  = add_news_features(test)

train = train[~train["target_1d"].isna()].copy()
val   = val[~val["target_1d"].isna()].copy()
test  = test[~test["target_1d"].isna()].copy()

y_train = train["target_1d"].values
y_val   = val["target_1d"].values
y_test  = test["target_1d"].values

price_features = [
    "ret_1d_lag", "ret_2d", "ret_3d", "ret_5d", "ret_10d",
    "overnight_ret", "intraday_ret",
    "RSI",
    "vol_3d", "vol_5d", "vol_10d",
    "range_vol",
    "turnover", "vol_z", "amihud"
]

news_momentum_features = []
for f in news_features:
    news_momentum_features += [
        f,                  # same-day sentiment
        f + "_lag1",
        f + "_roll3",
        f + "_roll5",
        f + "_vol5",
        f + "_shock",
        f + "_surprise3",
    ]

all_features = price_features + news_momentum_features

params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "device": "cuda",
    # "max_depth": 4,
    # "eta": 0.05,
    # "min_child_weight": 6,
    # "subsample": 0.8,
    # "colsample_bytree": 0.8,
    # "lambda": 4.0,
    "seed": 42,
}

# if no tuning, use these best params
if best_params_price is None:
    best_params_price = {'max_depth': 4, 'min_child_weight': 7.3505557352123505, 'eta': 0.010017838116843746, 'subsample': 0.9777856924636916, 'colsample_bytree': 0.6303799594249203, 'lambda': 0.008612782635599168, 'alpha': 3.7035837717863744}
if best_params_news is None:
    best_params_news = {'max_depth': 3, 'min_child_weight': 5.19102835385981, 'eta': 0.013971470063586327, 'subsample': 0.825393866505445, 'colsample_bytree': 0.6343691159723056, 'lambda': 1.3872924158083205e-05, 'alpha': 4.601521799703412}
if best_params_emb is None:
    best_params_emb = {
        "max_depth": 6,
        "min_child_weight": 7.0,
    "eta": 0.015,
    "subsample": 0.9,
    "colsample_bytree": 0.7,
    "lambda": 2.0,
    "alpha": 0.1,
}


def train_and_eval(name, features):
    print(f"\n===== {name} =====")

    X_train = train[features].values
    X_val   = val[features].values
    X_test  = test[features].values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val, label=y_val)
    dtest  = xgb.DMatrix(X_test, label=y_test)

    if name == "PRICE ONLY":
        tuned = best_params_price
    elif name == "PRICE + NEWS":
        tuned = best_params_news
    else:  # embeddings
        tuned = best_params_emb

    model = xgb.train(
        params={**params, **tuned},
        dtrain=dtrain,
        evals=[(dtrain, "train"), (dval, "val")],
        num_boost_round=2000,
        early_stopping_rounds=50,
        verbose_eval=50,
    )


    preds = model.predict(dtest)

    da   = (np.sign(preds) == np.sign(y_test)).mean()
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    ic   = spearmanr(preds, y_test).correlation

    print(f"DA:   {da:.4f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"IC:   {ic:.4f}")

    return {"DA": da, "RMSE": rmse, "IC": ic}, model




Embedding columns: 64


0        -0.003440
1         0.004439
2        -0.042474
3        -0.010513
4        -0.016844
            ...   
729654    0.016494
729655   -0.009168
729656    0.011382
729657   -0.007570
729658    0.000000
Name: target_1d, Length: 729659, dtype: float64

In [5]:
embedding_features = emb_cols

all_features_with_emb = price_features + news_momentum_features + embedding_features

metrics_price, model_price = train_and_eval("PRICE ONLY", price_features)
metrics_multi, model_multi = train_and_eval("PRICE + NEWS", all_features)
metrics_emb, model_emb = train_and_eval("PRICE + NEWS + EMBEDDINGS",
                                        all_features_with_emb)



print("\n=== SUMMARY ===")
print("Price only:", metrics_price)
print("Price + news:", metrics_multi)
print("Price + news + embeddings:", metrics_emb)


===== PRICE ONLY =====
[0]	train-rmse:0.02120	val-rmse:0.02424
[50]	train-rmse:0.02089	val-rmse:0.02423
[100]	train-rmse:0.02068	val-rmse:0.02424
[105]	train-rmse:0.02067	val-rmse:0.02424
DA:   0.5129
RMSE: 0.018404
IC:   0.0051

===== PRICE + NEWS =====
[0]	train-rmse:0.02120	val-rmse:0.02424
[50]	train-rmse:0.02092	val-rmse:0.02423
[94]	train-rmse:0.02074	val-rmse:0.02424
DA:   0.5135
RMSE: 0.018394
IC:   0.0024

===== PRICE + NEWS + EMBEDDINGS =====
[0]	train-rmse:0.02120	val-rmse:0.02424
[50]	train-rmse:0.02105	val-rmse:0.02423
[100]	train-rmse:0.02093	val-rmse:0.02423
DA:   0.5139
RMSE: 0.018362
IC:   0.0026

=== SUMMARY ===
Price only: {'DA': np.float64(0.5128881477146087), 'RMSE': np.float64(0.018403563678362853), 'IC': np.float64(0.005075677136039502)}
Price + news: {'DA': np.float64(0.5134654084520593), 'RMSE': np.float64(0.018393844273709915), 'IC': np.float64(0.0023797994580949055)}
Price + news + embeddings: {'DA': np.float64(0.513850248943693), 'RMSE': np.float64(0.018361

In [7]:
# save models
model_price.save_model("../results/models/xgb_price.model")
model_multi.save_model("../results/models/xgb_price_news.model")
model_emb.save_model("../results/models/xgb_price_news_emb.model")

/tmp/ipykernel_305238/1380731588.py:2: UserWarning: [19:12:22] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1763746907527/work/src/c_api/c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  model_price.save_model("../results/models/xgb_price.model")
/tmp/ipykernel_305238/1380731588.py:3: UserWarning: [19:12:22] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1763746907527/work/src/c_api/c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  model_multi.save_model("../results/models/xgb_price_news.model")
/tmp/ipykernel_305238/1380731588.py:4: UserWarning: [19:12:22] WARNING: /home/conda/feedstock_root/build_artifacts/xgboost-split_1763746907527/work/src/c_api/c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between f

In [74]:
rng = np.random.RandomState(42)
idx_shap = rng.choice(len(test), size=1000, replace=False)

X_test_price = test.iloc[idx_shap][price_features]
X_test_news  = test.iloc[idx_shap][all_features]
X_test_emb   = test.iloc[idx_shap][price_features + news_momentum_features + emb_cols]  # or your all_features_with_emb


In [79]:
def shap_for_model(model, X, feature_names, prefix):
    """
    model: xgboost.Booster (from xgb.train)
    X:     DataFrame (1000 x n_features)
    feature_names: list of feature names in order
    prefix: string used in saved filenames
    """
    print(f"\n=== SHAP for {prefix} ===")

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    # ---- 3.1 Summary (beeswarm) ----
    plt.figure(figsize=(10, 6))
    shap.summary_plot(
        shap_values,
        X,
        feature_names=feature_names,
        show=False
    )
    plt.title(f"SHAP Summary - {prefix}")
    plt.tight_layout()
    plt.savefig(f"shap_summary_{prefix}.png", dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved shap_summary_{prefix}.png")

    # ---- 3.2 Bar plot (mean |SHAP| importance) ----
    plt.figure(figsize=(8, 5))
    shap.summary_plot(
        shap_values,
        X,
        feature_names=feature_names,
        plot_type="bar",
        show=False
    )
    plt.title(f"SHAP Feature Importance - {prefix}")
    plt.tight_layout()
    plt.savefig(f"shap_bar_{prefix}.png", dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved shap_bar_{prefix}.png")

    # ---- 3.3 Dependence plots for top 5 features ----
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_idx = np.argsort(mean_abs_shap)[::-1][:5]
    top_features = [feature_names[i] for i in top_idx]

    for f in top_features:
        plt.figure(figsize=(6, 4))
        shap.dependence_plot(
            f,
            shap_values,
            X,
            feature_names=feature_names,
            show=False
        )
        plt.title(f"SHAP dependence - {f} ({prefix})")
        plt.tight_layout()
        fname = f"shap_dependence_{prefix}_{f}.png".replace("/", "_")
        plt.savefig(fname, dpi=200, bbox_inches="tight")
        plt.close()
        print(f"Saved {fname}")

    return shap_values


In [80]:
import shap
import matplotlib.pyplot as plt
# 1) PRICE ONLY
shap_price = shap_for_model(
    model_price,
    X_test_price,
    feature_names=price_features,
    prefix="price_only"
)

# 2) PRICE + NEWS (sentiment + momentum)
shap_news = shap_for_model(
    model_multi,
    X_test_news,
    feature_names=all_features,
    prefix="price_plus_news"
)

# 3) PRICE + NEWS + EMBEDDINGS
shap_emb = shap_for_model(
    model_emb,
    X_test_emb,
    feature_names=price_features + news_momentum_features + emb_cols,  # or your all_features_with_emb
    prefix="price_news_emb"
)



=== SHAP for price_only ===
Saved shap_summary_price_only.png
Saved shap_bar_price_only.png
Saved shap_dependence_price_only_vol_10d.png
Saved shap_dependence_price_only_RSI.png
Saved shap_dependence_price_only_ret_5d.png
Saved shap_dependence_price_only_overnight_ret.png
Saved shap_dependence_price_only_vol_3d.png

=== SHAP for price_plus_news ===
Saved shap_summary_price_plus_news.png
Saved shap_bar_price_plus_news.png
Saved shap_dependence_price_plus_news_vol_10d.png
Saved shap_dependence_price_plus_news_vol_3d.png
Saved shap_dependence_price_plus_news_RSI.png
Saved shap_dependence_price_plus_news_vol_5d.png
Saved shap_dependence_price_plus_news_ret_5d.png

=== SHAP for price_news_emb ===
Saved shap_summary_price_news_emb.png
Saved shap_bar_price_news_emb.png
Saved shap_dependence_price_news_emb_vol_10d.png
Saved shap_dependence_price_news_emb_vol_3d.png
Saved shap_dependence_price_news_emb_RSI.png
Saved shap_dependence_price_news_emb_overnight_ret.png
Saved shap_dependence_price_n

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

...

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

<Figure size 600x400 with 0 Axes>

In [70]:
def tune_xgb_price(n_trials=40):
    X_train = train[price_features].values
    X_val   = val[price_features].values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val,   label=y_val)

    def objective(trial):
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "device": "cuda",

            "max_depth": trial.suggest_int("max_depth", 3, 7),
            "min_child_weight": trial.suggest_float("min_child_weight", 2.0, 10.0),
            "eta": trial.suggest_float("eta", 0.01, 0.08, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

            "lambda": trial.suggest_float("lambda", 1e-6, 5.0, log=True),
            "alpha": trial.suggest_float("alpha", 0.0, 3.0),
        }

        model = xgb.train(
            params=params,
            dtrain=dtrain,
            evals=[(dval, "val")],
            num_boost_round=1500,
            early_stopping_rounds=50,
            verbose_eval=False,
        )

        preds = model.predict(dval)
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    print("BEST PRICE PARAMS:", study.best_params)
    return study.best_params

def tune_xgb_news(n_trials=40):
    X_train = train[all_features].values
    X_val   = val[all_features].values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val,   label=y_val)

    def objective(trial):
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "device": "cuda",

            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "min_child_weight": trial.suggest_float("min_child_weight", 3.0, 15.0),
            "eta": trial.suggest_float("eta", 0.008, 0.07, log=True),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

            "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
            "alpha": trial.suggest_float("alpha", 0.0, 5.0),
        }

        model = xgb.train(
            params=params,
            dtrain=dtrain,
            evals=[(dval, "val")],
            num_boost_round=2000,
            early_stopping_rounds=50,
            verbose_eval=False,
        )

        preds = model.predict(dval)
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    print("BEST NEWS PARAMS:", study.best_params)
    return study.best_params

def tune_xgb_embeddings(n_trials=40):
    feature_set = price_features + news_features + emb_cols

    X_train = train[feature_set].values
    X_val   = val[feature_set].values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val,   label=y_val)

    def objective(trial):
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            "tree_method": "hist",
            "device": "cuda",

            # embeddings require more depth
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0),

            # smaller learning rate
            "eta": trial.suggest_float("eta", 0.005, 0.05, log=True),

            # strong regularization
            "lambda": trial.suggest_float("lambda", 1e-6, 20.0, log=True),
            "alpha": trial.suggest_float("alpha", 0.0, 10.0),

            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        }

        model = xgb.train(
            params=params,
            dtrain=dtrain,
            evals=[(dval, "val")],
            num_boost_round=2500,
            early_stopping_rounds=50,
            verbose_eval=False,
        )

        preds = model.predict(dval)
        return np.sqrt(mean_squared_error(y_val, preds))

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    print("BEST EMBEDDING PARAMS:", study.best_params)
    return study.best_params


In [71]:
best_params_price = tune_xgb_price(n_trials=40)
best_params_news  = tune_xgb_news(n_trials=40)
best_params_emb   = tune_xgb_embeddings(n_trials=40)


[I 2025-11-24 18:02:47,228] A new study created in memory with name: no-name-ad76feea-92cc-4c30-8fe9-926a9cb9f631
[I 2025-11-24 18:02:47,491] Trial 0 finished with value: 0.024242329894341403 and parameters: {'max_depth': 6, 'min_child_weight': 8.049224969791796, 'eta': 0.010746556737319155, 'subsample': 0.8007208226291626, 'colsample_bytree': 0.7251023712692581, 'lambda': 0.00022325524163736518, 'alpha': 1.3996443182518488}. Best is trial 0 with value: 0.024242329894341403.
[I 2025-11-24 18:02:47,594] Trial 1 finished with value: 0.02425196762287709 and parameters: {'max_depth': 3, 'min_child_weight': 4.90038944298416, 'eta': 0.01886014961298444, 'subsample': 0.965422925187435, 'colsample_bytree': 0.8695627639487474, 'lambda': 0.0003500346929887863, 'alpha': 0.028504542355754725}. Best is trial 0 with value: 0.024242329894341403.
[I 2025-11-24 18:02:47,707] Trial 2 finished with value: 0.02425947718558182 and parameters: {'max_depth': 3, 'min_child_weight': 6.78886543124662, 'eta': 0.

BEST PRICE PARAMS: {'max_depth': 6, 'min_child_weight': 9.414942824935029, 'eta': 0.010103505214852922, 'subsample': 0.8443464008318593, 'colsample_bytree': 0.6405480132883279, 'lambda': 0.00025079252162247986, 'alpha': 0.4157661596047824}


[I 2025-11-24 18:02:55,425] A new study created in memory with name: no-name-e5b0b7a6-5c92-4a65-994a-da758a07b741
[I 2025-11-24 18:02:55,943] Trial 0 finished with value: 0.02426397704700138 and parameters: {'max_depth': 9, 'min_child_weight': 4.811173051620163, 'eta': 0.01543967931755844, 'subsample': 0.7981443829367514, 'colsample_bytree': 0.8330572403910593, 'lambda': 5.075339568016122e-07, 'alpha': 2.3422381793702023}. Best is trial 0 with value: 0.02426397704700138.
[I 2025-11-24 18:02:56,375] Trial 1 finished with value: 0.024243484438044433 and parameters: {'max_depth': 8, 'min_child_weight': 9.915144941925517, 'eta': 0.008027669762988814, 'subsample': 0.8290616096696015, 'colsample_bytree': 0.8028095773207905, 'lambda': 6.62076329511154e-06, 'alpha': 3.5614723507061843}. Best is trial 1 with value: 0.024243484438044433.
[I 2025-11-24 18:02:56,559] Trial 2 finished with value: 0.02429295546351273 and parameters: {'max_depth': 4, 'min_child_weight': 10.38541758646442, 'eta': 0.05

BEST NEWS PARAMS: {'max_depth': 7, 'min_child_weight': 8.763348041314794, 'eta': 0.00863982402503789, 'subsample': 0.9316621146144122, 'colsample_bytree': 0.7836354867728363, 'lambda': 4.179887618934322, 'alpha': 2.8674000588765827}


[I 2025-11-24 18:03:13,633] A new study created in memory with name: no-name-43c9b627-0faf-4e61-8388-c9b964132aa3
[I 2025-11-24 18:03:14,163] Trial 0 finished with value: 0.024240177482636967 and parameters: {'max_depth': 5, 'min_child_weight': 6.748791418762427, 'eta': 0.013933801171941554, 'lambda': 0.00011681503129749344, 'alpha': 0.7764751792257607, 'subsample': 0.9053851794294425, 'colsample_bytree': 0.6879772908354378}. Best is trial 0 with value: 0.024240177482636967.
[I 2025-11-24 18:03:15,173] Trial 1 finished with value: 0.02426047389262753 and parameters: {'max_depth': 12, 'min_child_weight': 9.463679719706086, 'eta': 0.016525786320577034, 'lambda': 8.534868927125276e-05, 'alpha': 4.871714060674586, 'subsample': 0.8095022946870151, 'colsample_bytree': 0.5519614789680397}. Best is trial 0 with value: 0.024240177482636967.
[I 2025-11-24 18:03:15,941] Trial 2 finished with value: 0.02423916900180365 and parameters: {'max_depth': 6, 'min_child_weight': 17.986298101953217, 'eta':

BEST EMBEDDING PARAMS: {'max_depth': 5, 'min_child_weight': 13.526369920891472, 'eta': 0.0060256031864824615, 'lambda': 0.7986934964547976, 'alpha': 1.9278630482302592, 'subsample': 0.9104958966643895, 'colsample_bytree': 0.644726102648078}


In [81]:
# save the best hyperparameters
import json
best_params = {
    "price": best_params_price,
    "news": best_params_news,
    "embeddings": best_params_emb,
}
with open("xgb_best_params.json", "w") as f:
    json.dump(best_params, f, indent=4)
print("Saved xgb_best_params.json")

Saved xgb_best_params.json


In [54]:
best_params_price = tune_xgb(price_features, n_trials=40, study_name="price_only")


[I 2025-11-24 17:39:49,931] A new study created in memory with name: price_only
[I 2025-11-24 17:39:50,369] Trial 0 finished with value: 0.024328638390818225 and parameters: {'max_depth': 10, 'min_child_weight': 7.069254097675014, 'eta': 0.03489041747330496, 'subsample': 0.6761458668409507, 'colsample_bytree': 0.9622993195892009, 'lambda': 0.002890203497697195, 'alpha': 4.883211019190558}. Best is trial 0 with value: 0.024328638390818225.
[I 2025-11-24 17:39:50,730] Trial 1 finished with value: 0.024363944083972258 and parameters: {'max_depth': 7, 'min_child_weight': 9.743755862216938, 'eta': 0.06301174897835478, 'subsample': 0.8096092738627941, 'colsample_bytree': 0.9562205078922115, 'lambda': 2.246455242204651, 'alpha': 4.30163819313885}. Best is trial 0 with value: 0.024328638390818225.
[I 2025-11-24 17:39:50,870] Trial 2 finished with value: 0.024251333954811168 and parameters: {'max_depth': 4, 'min_child_weight': 1.508297949307159, 'eta': 0.016754792237345658, 'subsample': 0.79853

==== BEST PARAMS ====
{'max_depth': 4, 'min_child_weight': 7.3505557352123505, 'eta': 0.010017838116843746, 'subsample': 0.9777856924636916, 'colsample_bytree': 0.6303799594249203, 'lambda': 0.008612782635599168, 'alpha': 3.7035837717863744}
Best Val RMSE: 0.024238141826730453


In [55]:
best_params_price = tune_xgb(price_features, n_trials=40, study_name="price_only")


[I 2025-11-24 17:40:15,088] A new study created in memory with name: price_only
[I 2025-11-24 17:40:15,243] Trial 0 finished with value: 0.024256549213690526 and parameters: {'max_depth': 3, 'min_child_weight': 5.581684257426199, 'eta': 0.026206110631614345, 'subsample': 0.8814074112871355, 'colsample_bytree': 0.7680749204566374, 'lambda': 2.7329256288237813e-08, 'alpha': 4.588138278408433}. Best is trial 0 with value: 0.024256549213690526.
[I 2025-11-24 17:40:15,720] Trial 1 finished with value: 0.024323808602480915 and parameters: {'max_depth': 10, 'min_child_weight': 8.987051177308874, 'eta': 0.03552521220769181, 'subsample': 0.6257540351429806, 'colsample_bytree': 0.7508299298607277, 'lambda': 1.926061265736266e-07, 'alpha': 3.2447708976732397}. Best is trial 0 with value: 0.024256549213690526.
[I 2025-11-24 17:40:15,856] Trial 2 finished with value: 0.024243205256046837 and parameters: {'max_depth': 3, 'min_child_weight': 5.19102835385981, 'eta': 0.013971470063586327, 'subsample':

==== BEST PARAMS ====
{'max_depth': 3, 'min_child_weight': 5.19102835385981, 'eta': 0.013971470063586327, 'subsample': 0.825393866505445, 'colsample_bytree': 0.6343691159723056, 'lambda': 1.3872924158083205e-05, 'alpha': 4.601521799703412}
Best Val RMSE: 0.024243205256046837


In [46]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

# -------------------------------------------------
# 0. Load clean data
# -------------------------------------------------
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"]).reset_index(drop=True)
val   = val.sort_values(["tic", "Date"]).reset_index(drop=True)
test  = test.sort_values(["tic", "Date"]).reset_index(drop=True)

news_features = ["mean_sentiment", "max_sentiment", "min_sentiment",
                 "sum_sentiment", "news_count"]


# -------------------------------------------------
# 1. Add past return + NEXT-DAY target (no leakage)
# -------------------------------------------------
def add_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["tic", "Date"]).copy()
    grp_close = df.groupby("tic")["Close"]

    # past 1-day close-to-close return (t vs t-1)
    df["ret_1d_raw"] = grp_close.pct_change(1)

    # target: next-day return (t -> t+1)
    # i.e. ret_1d_raw shifted one step forward
    df["target_1d"] = df.groupby("tic")["ret_1d_raw"].shift(-1)

    return df

train = add_target(train)
val   = add_target(val)
test  = add_target(test)


# -------------------------------------------------
# 2. Price-based short-horizon features (leak-free)
# -------------------------------------------------
def add_price_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["tic", "Date"]).copy()

    grp_close = df.groupby("tic")["Close"]
    grp_ret   = df.groupby("tic")["ret_1d_raw"]
    grp_vol   = df.groupby("tic")["Volume"]

    # --- Momentum (all past) ---
    df["ret_1d_lag"] = grp_close.pct_change(1)   # same as ret_1d_raw
    df["ret_2d"]     = grp_close.pct_change(2)
    df["ret_3d"]     = grp_close.pct_change(3)
    df["ret_5d"]     = grp_close.pct_change(5)
    df["ret_10d"]    = grp_close.pct_change(10)

    # --- Intraday & overnight (today vs yesterday) ---
    prev_close = grp_close.shift(1)
    df["overnight_ret"] = df["Open"] / prev_close - 1
    df["intraday_ret"]  = df["Close"] / df["Open"] - 1

    # --- Volatility (past returns) ---
    df["vol_3d"] = grp_ret.rolling(3).std().reset_index(0, drop=True)
    df["vol_5d"] = grp_ret.rolling(5).std().reset_index(0, drop=True)

    # --- Liquidity ---
    mean_vol = grp_vol.transform("mean")
    df["turnover"] = df["Volume"] / mean_vol

    # --- Time-series z-score of daily return (20d window) ---
    roll_mean_20 = grp_ret.rolling(20).mean().reset_index(0, drop=True)
    roll_std_20  = grp_ret.rolling(20).std().reset_index(0, drop=True)
    df["ret_1d_ts_z"] = (df["ret_1d_raw"] - roll_mean_20) / roll_std_20

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df

train = add_price_features(train)
val   = add_price_features(val)
test  = add_price_features(test)


# -------------------------------------------------
# 3. Same-day news sentiment + momentum features
#    (only up to day t, still leak-free for t+1)
# -------------------------------------------------
def add_news_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["tic", "Date"]).copy()
    grp = df.groupby("tic")

    for f in news_features:
        # Same-day sentiment (already in df[f])

        # Previous day's sentiment (t-1)
        df[f + "_lag1"] = grp[f].shift(1)

        # 3-day rolling mean up to today (t, t-1, t-2)
        df[f + "_roll3"] = grp[f].rolling(3).mean().reset_index(0, drop=True)

        # 5-day rolling mean up to today
        df[f + "_roll5"] = grp[f].rolling(5).mean().reset_index(0, drop=True)

        # 5-day rolling std (sentiment volatility)
        df[f + "_vol5"] = grp[f].rolling(5).std().reset_index(0, drop=True)

        # Sentiment shock: today's sentiment minus yesterday's
        df[f + "_shock"] = df[f] - df[f + "_lag1"]

        # Sentiment surprise vs 3-day average
        df[f + "_surprise3"] = df[f] - df[f + "_roll3"]

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return df

train = add_news_features(train)
val   = add_news_features(val)
test  = add_news_features(test)


# -------------------------------------------------
# 4. Drop rows with missing target
# -------------------------------------------------
train = train[~train["target_1d"].isna()].copy()
val   = val[~val["target_1d"].isna()].copy()
test  = test[~test["target_1d"].isna()].copy()

y_train = train["target_1d"].values
y_val   = val["target_1d"].values
y_test  = test["target_1d"].values


# -------------------------------------------------
# 5. Define feature sets
# -------------------------------------------------
price_features = [
    "ret_1d_lag", "ret_2d", "ret_3d", "ret_5d", "ret_10d",
    "intraday_ret", "overnight_ret",
    "vol_3d", "vol_5d",
    "turnover",
    "ret_1d_ts_z",
]

news_momentum_features = []
for f in news_features:
    news_momentum_features.extend([
        f,                  # same-day sentiment
        f + "_lag1",
        f + "_roll3",
        f + "_roll5",
        f + "_vol5",
        f + "_shock",
        f + "_surprise3",
    ])

all_features = price_features + news_momentum_features


# -------------------------------------------------
# 6. XGBoost GPU params
# -------------------------------------------------
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "device": "cuda",
    "max_depth": 4,
    "eta": 0.05,
    "min_child_weight": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 4.0,
    "alpha": 0.0,
    "seed": 42,
}


# -------------------------------------------------
# 7. Helper: train + evaluate
# -------------------------------------------------
def train_and_eval(name, feature_list):
    print(f"\n===== {name} =====")

    X_train = train[feature_list].values
    X_val   = val[feature_list].values
    X_test  = test[feature_list].values

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval   = xgb.DMatrix(X_val,   label=y_val)
    dtest  = xgb.DMatrix(X_test,  label=y_test)

    model = xgb.train(
        params=params,
        dtrain=dtrain,
        evals=[(dtrain, "train"), (dval, "val")],
        num_boost_round=2000,
        early_stopping_rounds=50,
        verbose_eval=50,
    )

    preds = model.predict(dtest)

    da   = (np.sign(preds) == np.sign(y_test)).mean()
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    ic   = spearmanr(preds, y_test).correlation

    print(f"Test DA:   {da:.4f}")
    print(f"Test RMSE: {rmse:.6f}")
    print(f"Test IC:   {ic:.4f}")

    return model, {"DA": da, "RMSE": rmse, "IC": ic}


# -------------------------------------------------
# 8. Run: price-only vs price+news
# -------------------------------------------------
model_price, metrics_price = train_and_eval("PRICE ONLY", price_features)
model_multi, metrics_multi = train_and_eval("PRICE + SAME-DAY NEWS", all_features)

print("\n=== SUMMARY COMPARISON ===")
print("Price only:     ", metrics_price)
print("Price + news:   ", metrics_multi)



===== PRICE ONLY =====
[0]	train-rmse:0.02118	val-rmse:0.02424
[50]	train-rmse:0.02067	val-rmse:0.02429
[57]	train-rmse:0.02064	val-rmse:0.02430
Test DA:   0.5131
Test RMSE: 0.018422
Test IC:   -0.0106

===== PRICE + SAME-DAY NEWS =====
[0]	train-rmse:0.02118	val-rmse:0.02424
[50]	train-rmse:0.02066	val-rmse:0.02429
[53]	train-rmse:0.02064	val-rmse:0.02429
Test DA:   0.5133
Test RMSE: 0.018413
Test IC:   -0.0050

=== SUMMARY COMPARISON ===
Price only:      {'DA': np.float64(0.5130805679604256), 'RMSE': np.float64(0.018421630843746917), 'IC': np.float64(-0.010631670442639013)}
Price + news:    {'DA': np.float64(0.5132729882062425), 'RMSE': np.float64(0.018413025775811182), 'IC': np.float64(-0.005029907600389059)}


In [33]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

# ----------------------------------------
# 0. Reload clean data (important!)
# ----------------------------------------
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')

train = train.sort_values(["tic", "Date"])
val   = val.sort_values(["tic", "Date"])
test  = test.sort_values(["tic", "Date"])


# ----------------------------------------
# 1. Add past-return series + target
# ----------------------------------------
def add_past_return_and_target(df):
    df = df.sort_values(["tic", "Date"]).copy()
    # past 1-day close-to-close return (t vs t-1)
    df["ret_1d_raw"] = df.groupby("tic")["Close"].pct_change(1)
    # target: next-day return (t -> t+1)
    df["target_1d_ahead"] = df.groupby("tic")["ret_1d_raw"].shift(-1)
    return df

train = add_past_return_and_target(train)
val   = add_past_return_and_target(val)
test  = add_past_return_and_target(test)

# ----------------------------------------
# 2. Feature engineering using ONLY past info
# ----------------------------------------
def add_features_no_leak(df):
    df = df.sort_values(["tic", "Date"]).copy()

    # --- Momentum from prices (all past) ---
    grp_close = df.groupby("tic")["Close"]
    df["ret_1d_lag"]  = grp_close.pct_change(1)   # t vs t-1
    df["ret_2d"]      = grp_close.pct_change(2)
    df["ret_3d"]      = grp_close.pct_change(3)
    df["ret_5d"]      = grp_close.pct_change(5)
    df["ret_10d"]     = grp_close.pct_change(10)

    # --- Intraday features (only up to today's close) ---
    # Need previous close per stock
    prev_close = grp_close.shift(1)
    df["overnight_ret"] = df["Open"] / prev_close - 1
    df["intraday_ret"]  = df["Close"] / df["Open"] - 1

    # --- RSI (14-day) based on past price moves ---
    delta = grp_close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    roll_up = gain.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    roll_down = loss.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    df["RSI"] = 100 - (100 / (1 + roll_up / roll_down))

    # --- Volatility based on PAST returns (ret_1d_raw) ---
    grp_ret = df.groupby("tic")["ret_1d_raw"]
    df["vol_3d"]  = grp_ret.rolling(3).std().reset_index(0, drop=True)
    df["vol_5d"]  = grp_ret.rolling(5).std().reset_index(0, drop=True)
    df["vol_10d"] = grp_ret.rolling(10).std().reset_index(0, drop=True)

    # --- High-Low range volatility (using today's H,L) ---
    df["range_vol"] = np.log(df["High"] / df["Low"])**2

    # --- Liquidity / volume-based (past) ---
    grp_vol = df.groupby("tic")["Volume"]
    mean_vol = grp_vol.transform("mean")
    std_vol  = grp_vol.transform("std")

    df["turnover"] = df["Volume"] / mean_vol
    df["vol_z"]    = (df["Volume"] - mean_vol) / std_vol

    # Amihud illiquidity (using PAST return, not target)
    df["amihud"] = (np.abs(df["ret_1d_raw"]) / df["Volume"]).replace([np.inf, -np.inf], np.nan)

    # Fill NaNs created by rolling / first rows
    df = df.fillna(0.0)

    return df

train = add_features_no_leak(train)
val   = add_features_no_leak(val)
test  = add_features_no_leak(test)

# ----------------------------------------
# 3. Keep only rows with non-NaN target
# ----------------------------------------
train = train[~train["target_1d_ahead"].isna()].copy()
val   = val[~val["target_1d_ahead"].isna()].copy()
test  = test[~test["target_1d_ahead"].isna()].copy()

y_train = train["target_1d_ahead"].values
y_val   = val["target_1d_ahead"].values
y_test  = test["target_1d_ahead"].values

# Feature list
momentum_features = [
    "ret_1d_lag", "ret_2d", "ret_3d", "ret_5d", "ret_10d",
    "overnight_ret", "intraday_ret",
    "RSI",
    "vol_3d", "vol_5d", "vol_10d",
    "range_vol",
    "turnover", "vol_z", "amihud"
] 

X_train = train[momentum_features].values
X_val   = val[momentum_features].values
X_test  = test[momentum_features].values

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)

# ----------------------------------------
# 4. XGBoost (GPU) training
# ----------------------------------------
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "device": "cuda",
    "max_depth": 4,
    "eta": 0.05,
    "min_child_weight": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 4.0,
    "alpha": 0.0,
    "seed": 42,
}

evals = [(dtrain, "train"), (dval, "val")]

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=1500,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50
)

# ----------------------------------------
# 5. Evaluate (now leakage-free)
# ----------------------------------------
y_pred_test = model.predict(dtest)

DA = (np.sign(y_pred_test) == np.sign(y_test)).mean()
RMSE = np.sqrt(mean_squared_error(y_test, y_pred_test))
IC = spearmanr(y_pred_test, y_test).correlation

print("\n=== Next-day return (NO LEAKAGE) ===")
print("Test Directional Accuracy:", DA)
print("Test RMSE:", RMSE)
print("Spearman IC:", IC)


[0]	train-rmse:0.02118	val-rmse:0.02424
[50]	train-rmse:0.02064	val-rmse:0.02427
[58]	train-rmse:0.02060	val-rmse:0.02428

=== Next-day return (NO LEAKAGE) ===
Test Directional Accuracy: 0.5134092858803627
Test RMSE: 0.01846526457309719
Spearman IC: 0.0073638713340866855


In [9]:
# print without truncation
pd.set_option('display.max_columns', None)
train

,Date,tic,Open,High,Low,Close,Volume,sales_growth_qoq,sales_growth_ttm,asset_growth,equity_growth,roa_ttm,roe_ttm,gross_margin_ttm,oper_margin_ttm,net_margin_ttm,log_mktcap,bm,earnings_yield,cf_yield,sales_yield,div_yield,leverage,current_ratio,cash_assets,accruals_ta,cpi,fedfunds,industrial_production,gdp,retail_sales,unemployment,t10y,t2y,t3m,aaa_yield,vix,sp500,yield_spread_10y_2y,return_next_day,mean_sentiment,max_sentiment,min_sentiment,sum_sentiment,news_count,pca_emb_0,pca_emb_1,pca_emb_2,pca_emb_3,pca_emb_4,pca_emb_5,pca_emb_6,pca_emb_7,pca_emb_8,pca_emb_9,pca_emb_10,pca_emb_11,pca_emb_12,pca_emb_13,pca_emb_14,pca_emb_15,pca_emb_16,pca_emb_17,pca_emb_18,pca_emb_19,pca_emb_20,pca_emb_21,pca_emb_22,pca_emb_23,pca_emb_24,pca_emb_25,pca_emb_26,pca_emb_27,pca_emb_28,pca_emb_29,pca_emb_30,pca_emb_31,pca_emb_32,pca_emb_33,pca_emb_34,pca_emb_35,pca_emb_36,pca_emb_37,pca_emb_38,pca_emb_39,pca_emb_40,pca_emb_41,pca_emb_42,pca_emb_43,pca_emb_44,pca_emb_45,pca_emb_46,pca_emb_47,pca_emb_48,pca_emb_49,pca_emb_50,pca_emb_51,pca_emb_52,pca_emb_53,pca_emb_54,pca_emb_55,pca_emb_56,pca_emb_57,pca_emb_58,pca_emb_59,pca_emb_60,pca_emb_61,pca_emb_62,pca_emb_63,ret_1d_raw,target_1d_ahead,ret_1d_lag,ret_2d,ret_3d,ret_5d,ret_10d,overnight_ret,intraday_ret,RSI,vol_3d,vol_5d,vol_10d,range_vol,turnover,vol_z,amihud
0,2016-01-04,A,37.978592,38.098833,37.312624,37.636356,3287300,0.020710,-0.002470,-0.309482,0.0,0.053617,0.0,0.560426,0.149827,0.099307,9.436385,0.0,0.031987,0.070834,0.322104,0.010593,0.442439,3.776639,0.300174,-0.065116,237.652,0.34,99.4391,19001.690,439466.0,4.8,2.24,1.02,0.26,4.00,20.70,2012.66,1.22,-0.003440,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.003440,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.009011,0.000000,0.000000,0.000000,0.000000,0.000435,1.621105,1.182416,0.000000e+00
1,2016-01-05,A,37.673370,37.876861,37.312638,37.506878,2587200,0.020710,-0.002470,-0.309482,0.0,0.053617,0.0,0.560426,0.149827,0.099307,9.436385,0.0,0.031987,0.070834,0.322104,0.010593,0.442439,3.776639,0.300174,-0.065116,237.652,0.34,99.4391,19001.690,439466.0,4.8,2.25,1.04,0.26,4.00,19.34,2016.71,1.21,0.004439,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.003440,0.004439,-0.003440,0.000000,0.000000,0.000000,0.000000,0.000983,-0.004419,0.000000,0.000000,0.000000,0.000000,0.000225,1.275856,0.525156,1.329719e-09
2,2016-01-06,A,37.220145,37.913860,37.044401,37.673370,2103600,0.020710,-0.002470,-0.309482,0.0,0.053617,0.0,0.560426,0.149827,0.099307,9.436385,0.0,0.031987,0.070834,0.322104,0.010593,0.442439,3.776639,0.300174,-0.065116,237.652,0.34,99.4391,19001.690,439466.0,4.8,2.18,0.99,0.26,4.00,20.59,1990.26,1.19,-0.042474,0.787009,0.835513,0.738505,1.574018,2.0,4.772681,2.465096,-1.987462,0.205819,4.225852,-1.834629,-2.518933,-0.924245,-1.239315,0.598441,0.169646,-0.141145,1.358956,-0.562618,0.534386,0.658163

In [29]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr

# ============================================================
# 0. Reload CLEAN data
# ============================================================
train = pd.read_parquet('../data/model/final_train.parquet')
val   = pd.read_parquet('../data/model/final_val.parquet')
test  = pd.read_parquet('../data/model/final_test.parquet')
# merge train with sp500_companies.csv to get sector info
sp500_info = pd.read_csv('../data/sp500_companies.csv')  # columns

train = train.sort_values(["tic", "Date"]).reset_index(drop=True)
val   = val.sort_values(["tic", "Date"]).reset_index(drop=True)
test  = test.sort_values(["tic", "Date"]).reset_index(drop=True)

# merge with sp500_info to get sector info
train = train.merge(sp500_info[['ticker', 'sector']], left_on='tic', right_on='ticker', how='left')
val   = val.merge(sp500_info[['ticker', 'sector']], left_on='tic', right_on='ticker', how='left')
test  = test.merge(sp500_info[['ticker', 'sector']], left_on='tic', right_on='ticker', how='left')
train[train["sector"] == "Information Technology"]
val[val["sector"] == "Information Technology"]
test[test["sector"] == "Information Technology"]

# ============================================================
# 1. Add past-return series + next-day target (NO leakage)
# ============================================================
def add_past_return_and_target(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["tic", "Date"]).copy()
    grp_close = df.groupby("tic")["Close"]

    # past 1-day close-to-close return at time t (uses t and t-1)
    df["ret_1d_raw"] = grp_close.pct_change(1)

    # target: next-day return (t -> t+1), i.e. ret_1d_raw shifted one step forward
    df["target_1d_ahead"] = df.groupby("tic")["ret_1d_raw"].shift(-1)

    return df

train = add_past_return_and_target(train)
val   = add_past_return_and_target(val)
test  = add_past_return_and_target(test)

# ============================================================
# 2. Feature engineering using ONLY past info
# ============================================================
def add_short_horizon_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(["tic", "Date"]).copy()

    grp_close = df.groupby("tic")["Close"]
    grp_ret   = df.groupby("tic")["ret_1d_raw"]
    grp_vol   = df.groupby("tic")["Volume"]

    # -------------------------
    # 2.1 Momentum features
    # -------------------------
    df["ret_1d_lag"] = grp_close.pct_change(1)    # same as ret_1d_raw but keep separate
    df["ret_2d"]     = grp_close.pct_change(2)
    df["ret_3d"]     = grp_close.pct_change(3)
    df["ret_5d"]     = grp_close.pct_change(5)
    df["ret_10d"]    = grp_close.pct_change(10)

    # -------------------------
    # 2.2 Intraday features (today only)
    # -------------------------
    prev_close = grp_close.shift(1)
    df["overnight_ret"] = df["Open"] / prev_close - 1
    df["intraday_ret"]  = df["Close"] / df["Open"] - 1

    # -------------------------
    # 2.3 RSI (14-day) + Fisher transform of RSI
    # -------------------------
    delta = grp_close.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)

    roll_up   = gain.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)
    roll_down = loss.groupby(df["tic"]).rolling(14).mean().reset_index(0, drop=True)

    rs = roll_up / roll_down
    df["RSI"] = 100 - (100 / (1 + rs))

    # Fisher transform of RSI: map RSI from [0,100] -> [-1,1] then apply Fisher
    x = (df["RSI"] / 100.0) * 2 - 1  # 0..100 -> -1..1
    x = x.clip(-0.999, 0.999)
    df["Fisher_RSI"] = 0.5 * np.log((1 + x) / (1 - x))

    # -------------------------
    # 2.4 Volatility (past returns based)
    # -------------------------
    df["vol_3d"]  = grp_ret.rolling(3).std().reset_index(0, drop=True)
    df["vol_5d"]  = grp_ret.rolling(5).std().reset_index(0, drop=True)
    df["vol_10d"] = grp_ret.rolling(10).std().reset_index(0, drop=True)

    # High-low "range" volatility proxy
    df["range_vol"] = np.log(df["High"] / df["Low"]) ** 2

    # Parkinson volatility
    df["parkinson_vol"] = df["range_vol"] / (4 * np.log(2))

    # Garman-Klass volatility
    with np.errstate(divide='ignore', invalid='ignore'):
        log_hl = np.log(df["High"] / df["Low"])
        log_co = np.log(df["Close"] / df["Open"])
    df["gk_vol"] = 0.5 * (log_hl ** 2) - (2 * np.log(2) - 1) * (log_co ** 2)

    # -------------------------
    # 2.5 Liquidity / volume features
    # -------------------------
    mean_vol = grp_vol.transform("mean")
    std_vol  = grp_vol.transform("std")

    df["turnover"] = df["Volume"] / mean_vol
    df["vol_z"]    = (df["Volume"] - mean_vol) / std_vol

    # Amihud illiquidity: use PAST return (ret_1d_raw), not target
    amihud = (np.abs(df["ret_1d_raw"]) / (df["Volume"].replace(0, np.nan)))
    df["amihud"] = amihud.replace([np.inf, -np.inf], np.nan)

    # -------------------------
    # 2.6 Time-series z-scored returns (20-day)
    # -------------------------
    roll_mean_20 = grp_ret.rolling(20).mean().reset_index(0, drop=True)
    roll_std_20  = grp_ret.rolling(20).std().reset_index(0, drop=True)
    df["ret_1d_ts_z"] = (df["ret_1d_raw"] - roll_mean_20) / roll_std_20

    # -------------------------
    # 2.7 Rolling skewness / kurtosis of returns (20-day)
    # -------------------------
    df["ret_skew_20d"] = grp_ret.rolling(20).skew().reset_index(0, drop=True)
    df["ret_kurt_20d"] = grp_ret.rolling(20).kurt().reset_index(0, drop=True)

    # -------------------------
    # 2.8 Intraday shadow ranges (candlestick)
    # -------------------------
    high = df["High"]
    low  = df["Low"]
    op   = df["Open"]
    cl   = df["Close"]

    upper_body = np.maximum(op, cl)
    lower_body = np.minimum(op, cl)

    # Normalize by Close to make them scale-free
    df["upper_shadow"] = (high - upper_body) / cl.replace(0, np.nan)
    df["lower_shadow"] = (lower_body - low) / cl.replace(0, np.nan)

    # -------------------------
    # 2.9 VWAP / typical price deviation proxies
    # (we don't have real intraday VWAP, so use typical price)
    # -------------------------
    typical_price = (df["High"] + df["Low"] + df["Close"]) / 3.0
    df["typical_price"] = typical_price
    df["close_vs_typical"] = (df["Close"] - typical_price) / typical_price.replace(0, np.nan)

    # -------------------------
    # 2.10 Sector-relative features (if sector exists)
    # -------------------------
    if "sector" in df.columns:
        # mean return within sector per day (using past ret_1d_raw)
        sector_ret = df.groupby(["Date", "sector"])["ret_1d_raw"].transform("mean")
        df["ret_vs_sector"] = df["ret_1d_raw"] - sector_ret

        # volume vs sector mean
        sector_vol = df.groupby(["Date", "sector"])["Volume"].transform("mean")
        df["vol_vs_sector"] = df["Volume"] / sector_vol.replace(0, np.nan)
    else:
        df["ret_vs_sector"] = 0.0
        df["vol_vs_sector"] = 0.0

    # -------------------------
    # 2.11 Cross-sectional ranks per day (VERY powerful)
    # -------------------------
    # Choose a subset of raw features to rank cross-sectionally
    cs_cols = [
        "ret_1d_lag",
        "ret_5d",
        "vol_5d",
        "turnover",
        "amihud",
        "Fisher_RSI"
    ]

    for col in cs_cols:
        # rank within each day (0..1)
        df[col + "_cs_rank"] = df.groupby("Date")[col].rank(pct=True)

    # -------------------------
    # 2.12 Final clean-up: replace inf/NaN with 0
    # -------------------------
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0.0)

    return df

train = add_short_horizon_features(train)
val   = add_short_horizon_features(val)
test  = add_short_horizon_features(test)

# ============================================================
# 3. Drop rows with missing target
# ============================================================
train = train[~train["target_1d_ahead"].isna()].copy()
val   = val[~val["target_1d_ahead"].isna()].copy()
test  = test[~test["target_1d_ahead"].isna()].copy()

y_train = train["target_1d_ahead"].values
y_val   = val["target_1d_ahead"].values
y_test  = test["target_1d_ahead"].values

# ============================================================
# 4. Define final feature list
# ============================================================
feature_cols = [
    # momentum
    "ret_1d_lag", "ret_2d", "ret_3d", "ret_5d", "ret_10d",
    "ret_1d_ts_z",

    # intraday / RSI
    "overnight_ret", "intraday_ret",
    "RSI", "Fisher_RSI",

    # volatility
    "vol_3d", "vol_5d", "vol_10d",
    "range_vol", "parkinson_vol", "gk_vol",

    # liquidity
    "turnover", "vol_z", "amihud",

    # skew/kurt
    "ret_skew_20d", "ret_kurt_20d",

    # shadows / vwap-ish
    "upper_shadow", "lower_shadow",
    "close_vs_typical",

    # sector-relative
    "ret_vs_sector", "vol_vs_sector",

    # cross-sectional ranks
    "ret_1d_lag_cs_rank",
    "ret_5d_cs_rank",
    "vol_5d_cs_rank",
    "turnover_cs_rank",
    "amihud_cs_rank",
    "Fisher_RSI_cs_rank",
]

X_train = train[feature_cols].values
X_val   = val[feature_cols].values
X_test  = test[feature_cols].values

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)

# ============================================================
# 5. Train GPU XGBoost for next-day return
# ============================================================
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "device": "cuda",
    "max_depth": 4,
    "eta": 0.05,
    "min_child_weight": 8,   # a bit stronger regularization
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "lambda": 5.0,
    "alpha": 0.0,
    "seed": 42,
}

evals = [(dtrain, "train"), (dval, "val")]

model = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=2000,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=50
)

# ============================================================
# 6. Evaluate
# ============================================================
y_pred_test = model.predict(dtest)

DA = (np.sign(y_pred_test) == np.sign(y_test)).mean()
RMSE = np.sqrt(mean_squared_error(y_test, y_pred_test))
IC = spearmanr(y_pred_test, y_test).correlation

print("\n=== Next-day return — advanced short-horizon features ===")
print("Test Directional Accuracy:", DA)
print("Test RMSE:", RMSE)
print("Spearman IC:", IC)


[0]	train-rmse:0.02117	val-rmse:0.02424
[50]	train-rmse:0.02053	val-rmse:0.02424
[74]	train-rmse:0.02040	val-rmse:0.02425

=== Next-day return — advanced short-horizon features ===
Test Directional Accuracy: 0.5121104492210988
Test RMSE: 0.018441718278141846
Spearman IC: -0.002359450945909622


In [16]:
sp500_info

,ticker,company_name,sector,subsector,cik
0,MMM,3M,Industrials,Industrial Conglomerates,66740
1,AOS,A. O. Smith,Industrials,Building Products,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1800
3,ABBV,AbbVie,Health Care,Biotechnology,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,1467373
...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,1524472
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,1041061
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,877212
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,1136869


In [17]:
sp500_info.groupby('sector').size()

sector
Communication Services    24
Consumer Discretionary    51
Consumer Staples          38
Energy                    22
Financials                74
Health Care               60
Industrials               78
Information Technology    68
Materials                 26
Real Estate               31
Utilities                 31
dtype: int64